# Transfermarkt HTML analysis

This is just a notebook for analyzing the HTML structuture of Transfermarkt and testing diffenent scripts to obtain the required information

## Extracting Club Information from Transfermarkt Premier League Table

Analyzing English Premier League table for season 2022-2023

https://www.transfermarkt.com/wettbewerb/startseite/wettbewerb/ES1/plus/?saison_id=2022

The goal here is to obatain club's name and club's url

### Getting raw HTML squad table

In [2]:
import requests
from bs4 import BeautifulSoup

# TM Premier League 2023-2024 URL
url = "https://www.transfermarkt.com/premier-league/startseite/wettbewerb/GB1/plus/?saison_id=2023"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.content, "html.parser")

table = soup.find("table", {"class": "items"})

rows = table.find_all("tr", {"class": ["odd", "even"]})

# Printing the first row to inspect the structure
row = rows[0]
print(row.prettify())

<tr class="odd">
 <td class="zentriert no-border-rechts">
  <a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City">
   <img alt="Manchester City" class="tiny_wappen" src="https://tmssl.akamaized.net//images/wappen/tiny/281.png?lm=1467356331" title="Manchester City"/>
  </a>
 </td>
 <td class="hauptlink no-border-links">
  <a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City">
   Manchester City
  </a>
  <a href="#">
   <img alt="English Champion 22/23" class="tabelle-erfolg" src="https://tmssl.akamaized.net//images/erfolge/mini/12.png?lm=1520606997" title="English Champion 22/23"/>
  </a>
  <a href="#">
   <img alt="FA Cup Winner 22/23" class="tabelle-erfolg" src="https://tmssl.akamaized.net//images/erfolge/mini/29.png?lm=1520606999" title="FA Cup Winner 22/23"/>
  </a>
  <a href="#">
   <img alt="Champions League Winner 22/23" class="tabelle-erfolg" src="https://tmssl.akamaized.net//images/erfolge/mini/4.png?lm=1520606

### Getting club link and name

In [3]:
club_link = row.find("a", href=lambda h: h and "/startseite/verein/" in h)
if club_link:
    print(club_link["href"])

/manchester-city/startseite/verein/281/saison_id/2023


**NAME**

In [4]:
print(club_link["href"])

club_name = club_link.get("title", "Unknown Club")
print(club_name)

/manchester-city/startseite/verein/281/saison_id/2023
Manchester City


**URL**

In [5]:
squad_url = club_link["href"].replace("/startseite/", "/kader/") + f"/saison_id/2023"
print(squad_url)

/manchester-city/kader/verein/281/saison_id/2023/saison_id/2023


In [6]:
club_links = []
club_links.append({
                "club_name": club_name,
                "squad_url": "https://www.transfermarkt.com" + squad_url if not squad_url.startswith("http") else squad_url
            })

for club_info in club_links:
    print(club_info["club_name"], club_info["squad_url"])

Manchester City https://www.transfermarkt.com/manchester-city/kader/verein/281/saison_id/2023/saison_id/2023


For all teams:

In [7]:
for row in rows:
    club_link = row.find("a", href=lambda h: h and "/startseite/verein/" in h)
    if club_link:
        club_href = club_link.get("href")
        club_name = club_link.get("title", "Unknown Club")
        squad_url = club_href.replace("/startseite/", "/kader/") + f"/saison_id/2023"
        club_links.append({
                "club_name": club_name,
                "squad_url": "https://www.transfermarkt.com" + squad_url if not squad_url.startswith("http") else squad_url
            })

Printing everything:

In [8]:
for club_info in club_links:
    print(club_info["club_name"], club_info["squad_url"])

Manchester City https://www.transfermarkt.com/manchester-city/kader/verein/281/saison_id/2023/saison_id/2023
Manchester City https://www.transfermarkt.com/manchester-city/kader/verein/281/saison_id/2023/saison_id/2023
Arsenal FC https://www.transfermarkt.com/fc-arsenal/kader/verein/11/saison_id/2023/saison_id/2023
Chelsea FC https://www.transfermarkt.com/fc-chelsea/kader/verein/631/saison_id/2023/saison_id/2023
Liverpool FC https://www.transfermarkt.com/fc-liverpool/kader/verein/31/saison_id/2023/saison_id/2023
Tottenham Hotspur https://www.transfermarkt.com/tottenham-hotspur/kader/verein/148/saison_id/2023/saison_id/2023
Manchester United https://www.transfermarkt.com/manchester-united/kader/verein/985/saison_id/2023/saison_id/2023
Aston Villa https://www.transfermarkt.com/aston-villa/kader/verein/405/saison_id/2023/saison_id/2023
Newcastle United https://www.transfermarkt.com/newcastle-united/kader/verein/762/saison_id/2023/saison_id/2023
Brighton & Hove Albion https://www.transferma

## Extracting Player information from club's table

*For this example I'm using Arsenal*

We want to obtain this information from player:
- Name
- ID
- url
- age
- position
- date of birth
- club
- league_id
- season

In [9]:
arsenal_url = "https://www.transfermarkt.com/fc-arsenal/kader/verein/11/saison_id/2023/saison_id/2023/plus/1"

arsenal_response = requests.get(arsenal_url, headers=headers)
arsenal_soup = BeautifulSoup(arsenal_response.content, "html.parser")

table = arsenal_soup.find("table", {"class": "items"})
if not table:
    print("Squad table not found in Arsenal page")
else:
    print("Squad table found in Arsenal page")
    arsenal_rows = table.find_all("tr", {"class": ["odd", "even"]})
    print(f"Found {len(arsenal_rows)} player rows in Arsenal squad table")

Squad table found in Arsenal page
Found 40 player rows in Arsenal squad table


In [10]:
arsenal_player_row = arsenal_rows[0]
print(arsenal_player_row.prettify())

<tr class="odd">
 <td class="zentriert rueckennummer bg_Torwart" title="Goalkeeper">
  <div class="rn_nummer">
   22
  </div>
 </td>
 <td class="posrela">
  <table class="inline-table">
   <tr>
    <td rowspan="2">
     <img alt="David Raya" class="bilderrahmen-fixed lazy lazy" data-src="https://img.a.transfermarkt.technology/portrait/medium/262749-1668168018.jpg?lm=1" src="data:image/gif;base64,R0lGODlhAQABAIAAAMLCwgAAACH5BAAAAAAALAAAAAABAAEAAAICRAEAOw==" title="David Raya">
     </img>
    </td>
    <td class="hauptlink">
     <a href="/david-raya/profil/spieler/262749">
      David Raya
     </a>
    </td>
   </tr>
   <tr>
    <td>
     Goalkeeper
    </td>
   </tr>
  </table>
 </td>
 <td class="zentriert">
  15/09/1995 (28)
 </td>
 <td class="zentriert">
  <img alt="Spain" class="flaggenrahmen" src="https://tmssl.akamaized.net//images/flagge/verysmall/157.png?lm=1520611569" title="Spain"/>
 </td>
 <td class="zentriert">
  <a href="/fc-arsenal/startseite/verein/11" title="Arsenal FC

In [11]:
test_player_link = arsenal_player_row.find("a", href=lambda h: h and "/profil/spieler/" in h)
player_name = test_player_link.get_text(strip=True)
player_id = test_player_link["href"].split("/profil/spieler/")[1].split("/")[0]
print(player_name + " - " + test_player_link["href"] + " - ID: " + player_id)

David Raya - /david-raya/profil/spieler/262749 - ID: 262749


Getting Date of birth and age

In [25]:
zentriert_cells = [td for td in arsenal_player_row.find_all("td") 
                   if "zentriert" in (td.get("class") or [])]

print(f"Total cells zentriert: {len(zentriert_cells)}")
for i, cell in enumerate(zentriert_cells):
    print(f"[{i}]: {cell.get_text(strip=True)}")

dob_text = zentriert_cells[1].get_text(strip=True)
print(dob_text) # Must print something like "15.09.1998"

Total cells zentriert: 8
[0]: 22
[1]: 15/09/1995 (28)
[2]: 
[3]: 
[4]: 1,83m
[5]: right
[6]: 04/07/2024
[7]: 
15/09/1995 (28)


In [ ]:
import re
from datetime import datetime,date

season_start_date = date(2023, 8, 1)

formats = [
        "%b %d, %Y",   # Jan 15, 2000
        "%d.%m.%Y",    # 15.01.2000
        "%d/%m/%Y",    # 15/01/2000  
        "%m/%d/%Y",    # 01/15/2000 
        "%Y-%m-%d",    # 2000-01-15  
    ]

match_dob = re.match(r"(\d{2}/\d{2}/\d{4})", dob_text)
print(match_dob.group(1) if match_dob else "DOB not found in expected format")

if match_dob:
    dob_str = match_dob.group(1)

    for fmt in formats:
        try:
            test_player_dob = datetime.strptime(dob_str.strip(), fmt).date()
            print(f"Parsed DOB: {test_player_dob} using format: {fmt}")
            break
        except ValueError:
            continue

if test_player_dob:
    test_player_age_at_season = (season_start_date - test_player_dob).days // 365
    print(f"Player DOB: {test_player_dob}, Age at season start: {test_player_age_at_season}")
else:
    print("Failed to parse player DOB")



15/09/1995
Parsed DOB: 1995-09-15 using format: %d/%m/%Y
Player DOB: 1995-09-15, Age at season start: 27


: 